In [3]:
#########################################[ Li2S / Li3N / LiTFSI center placement (auto-height) ]#########################################
import os
from typing import Optional, List, Tuple
import numpy as np
from ase import Atoms, Atom
from ase.db import connect
from ase.io import read
from ase.geometry import get_distances  # MIC-aware pair距离矩阵
# ------------------------------------------------------------
# 输入/输出
IN_DB = 'Surfaces.db'         # 输入底物数据库
OUT_DBS = {                   # 不同吸附质输出到不同库
    'Li2S':   'Surfaces_Li2S.db',
    'Li3N':   'Surfaces_Li3N.db',
    'LiTFSI': 'Surfaces_LiTFSI.db',
}
# 初始放置高度（Å）：若为 None 则用默认；最终会经“自动贴近”微调
INIT_HEIGHT = {
    'Li2S':   1.4,
    'Li3N':   1.8,  # Li3N 更“紧凑/反应活泼”，给低一点起始更容易压到合理接触
    'LiTFSI': 3.5,  # 体积大、柔性大，起点留高一点
}
# 平面内旋转（度）
ROT_Z_DEG = {
    'Li2S':   0.0,
    'Li3N':   0.0,
    'LiTFSI': 0.0,
}
# 分子文件候选名（按顺序尝试读取）
CANDIDATE_FILES = {
    'Li2S':   ['Li2S.xyz', 'Li2S.traj', 'Li2S.pdb', 'Li2S.mol'],
    'Li3N':   ['Li3N.xyz', 'Li3N.traj', 'Li3N.pdb', 'Li3N.mol'],
    'LiTFSI': ['LiTFSI.xyz', 'LiTFSI.traj', 'LiTFSI.pdb', 'LiTFSI.mol'],
}
# 目标放置点（分数坐标）
TARGET_FRAC = (0.5, 0.5)

# —— 自动贴近参数（按吸附体可微调）——
# clearance = max( d_ij - (r_vdW(i)+r_vdW(j) - fudge) ) >= 0
VdW = {  # Bondi/常用表 (Å)
    'H':1.20, 'C':1.70, 'N':1.55, 'O':1.52, 'F':1.47,
    'Li':1.82, 'Na':2.27, 'S':1.80, 'Cl':1.75, 'P':1.80
}
AUTO_CLOSE_CFG = {
    'Li2S':   dict(fudge=0.25, step=0.10, min_height=1.0, max_iter=60),
    'Li3N':   dict(fudge=0.20, step=0.05, min_height=0.8, max_iter=80),   # 更贴近一点 + 更细步长
    'LiTFSI': dict(fudge=0.30, step=0.10, min_height=1.2, max_iter=80),
}
# ------------------------------------------------------------

def frac_to_xy(atoms: Atoms, fx: float, fy: float) -> Tuple[float, float]:
    """分数坐标 -> 绝对 x–y（考虑非正交晶胞）"""
    a, b, _ = atoms.get_cell()
    r = fx * a + fy * b
    return float(r[0]), float(r[1])

def load_adsorbate_from_candidates(names: List[str]) -> Optional[Atoms]:
    """按候选文件列表读取 Atoms；若全不存在返回 None"""
    for fn in names:
        if os.path.exists(fn):
            return read(fn)
    return None

def build_Li2S_guess() -> Atoms:
    """极简 Li2S 占位：S 在原点，2 个 Li 沿 x 轴对称，Li–S ~2.3 Å（建议提供优化结构文件替换）"""
    d = 2.3
    atoms = Atoms()
    atoms.append(Atom('S', (0.0, 0.0, 0.0)))
    atoms.append(Atom('Li', ( d, 0.0, 0.0)))
    atoms.append(Atom('Li', (-d, 0.0, 0.0)))
    return atoms

def build_Li3N_guess() -> Atoms:
    """极简 Li3N 占位：N 在中心，3 个 Li 等边三角形，Li–N ~2.0 Å（建议提供优化结构文件替换）"""
    d = 2.0
    li_xy = [
        ( d, 0.0),
        (-0.5*d,  (np.sqrt(3)/2)*d),
        (-0.5*d, -(np.sqrt(3)/2)*d),
    ]
    atoms = Atoms()
    atoms.append(Atom('N', (0.0, 0.0, 0.0)))
    for x, y in li_xy:
        atoms.append(Atom('Li', (x, y, 0.0)))
    return atoms

def prepare_adsorbate(kind: str) -> Optional[Atoms]:
    """优先读文件；Li2S/Li3N 找不到则给一个简化占位；LiTFSI 找不到则跳过"""
    ads = load_adsorbate_from_candidates(CANDIDATE_FILES[kind])
    if ads is not None:
        return ads
    if kind == 'Li2S':
        print("[Info] 未找到 Li2S 文件，使用简化几何作为占位。")
        return build_Li2S_guess()
    if kind == 'Li3N':
        print("[Info] 未找到 Li3N 文件，使用简化几何作为占位。")
        return build_Li3N_guess()
    print("[Warn] 未找到 LiTFSI 结构文件（例如 LiTFSI.xyz）—将跳过 LiTFSI 的生成。")
    return None

def rotate_about_axis(ads: Atoms, angle_deg: float, axis='z', center='COM'):
    """围绕给定轴旋转分子（度）。axis 可为 'x'/'y'/'z' 或 3D 向量。"""
    if not angle_deg:
        return
    ads.rotate(angle_deg, v=axis, center=center, rotate_cell=False)

def center_xy_and_raise_z(slab: Atoms, ads: Atoms, height: float, fx=0.5, fy=0.5):
    """把 ads 的 COM 放到分数 (fx,fy)，并把最低 z 提到 slab 顶层 z + height；返回平移后的坐标副本"""
    A = ads.copy()
    # 把分子最低 z 移到 0
    minz = A.positions[:, 2].min()
    A.positions[:, 2] -= minz
    # 底物顶层 z
    top_z = slab.positions[:, 2].max()
    # 先把 COM 对齐到原点，再整体平移到目标 xy
    com = A.get_center_of_mass()
    A.positions[:, :2] -= com[:2]
    cx, cy = frac_to_xy(slab, fx, fy)
    A.positions[:, 2] += (top_z + height)
    A.positions[:, 0] += cx
    A.positions[:, 1] += cy
    return A

def min_clearance_to_vdw(slab: Atoms, ads: Atoms, pbc=(True, True, False), fudge: float = 0.25) -> float:
    """
    计算“最小余量” = min_ij { d_ij – [r_vdW(i)+r_vdW(j) – fudge] }。
    >0 表示还未接触（可以继续压低）；<=0 表示已触到/穿插。
    """
    # ASE 的 get_distances 支持 MIC（非正交+PBC）
    D = get_distances(ads.positions, slab.positions, cell=slab.cell, pbc=pbc)[1]  # 只取距离矩阵
    sym_ads = ads.get_chemical_symbols()
    sym_slab = slab.get_chemical_symbols()
    # 构造 (r_i + r_j - fudge) 阈值矩阵
    rv_ads = np.array([VdW.get(s, 1.7) for s in sym_ads])[:, None]  # 未知元素用 1.7 Å 兜底
    rv_slab = np.array([VdW.get(s, 1.7) for s in sym_slab])[None, :]
    thresh = rv_ads + rv_slab - float(fudge)
    margin = D - thresh
    return float(np.min(margin))

def auto_height_close(slab: Atoms, ads0: Atoms, fx=0.5, fy=0.5,
                      init_h: float = 3.0, fudge: float = 0.25,
                      step: float = 0.1, min_height: float = 1.0,
                      max_iter: int = 60, rot_z_deg: float = 0.0) -> Tuple[Atoms, float]:
    """
    从 init_h 开始，逐步向下（-step）压，直到最小“vdW余量”<=0；回退一步作为最终高度。
    返回：(放置后的 adsorbate，最终高度)
    """
    ads = ads0.copy()
    rotate_about_axis(ads, rot_z_deg, axis='z', center='COM')

    h = float(init_h)
    best_ads = center_xy_and_raise_z(slab, ads, h, fx, fy)
    best_h = h

    for k in range(max_iter):
        placed = center_xy_and_raise_z(slab, ads, h, fx, fy)
        m = min_clearance_to_vdw(slab, placed, pbc=slab.get_pbc(), fudge=fudge)
        if m > 0 and h - step >= min_height:
            # 还能更近：继续降低
            best_ads, best_h = placed, h
            h -= step
            continue
        # 已经触到/穿插（m<=0）或到达下限：回退并停止
        if m <= 0:
            # 用上一次“未触碰”的安全高度
            return best_ads, best_h
        else:
            return placed, h
    return best_ads, best_h  # 达到迭代上限

def place_adsorbate_centered_auto(slab: Atoms, ads_in: Atoms, kind: str,
                                  fx=0.5, fy=0.5, rot_z_deg: float = 0.0) -> Tuple[Atoms, float]:
    """自动高度贴近并合并到 slab；返回 (新slab, 实际高度)"""
    cfg = AUTO_CLOSE_CFG[kind]
    init_h = INIT_HEIGHT[kind] if INIT_HEIGHT[kind] is not None else 3.0
    ads_placed, final_h = auto_height_close(
        slab, ads_in, fx=fx, fy=fy, init_h=init_h,
        fudge=cfg['fudge'], step=cfg['step'],
        min_height=cfg['min_height'], max_iter=cfg['max_iter'],
        rot_z_deg=rot_z_deg
    )
    slab_new = slab.copy()
    slab_new.extend(ads_placed)
    return slab_new, final_h

def main():
    db_in = connect(IN_DB)

    # 逐一处理三种吸附质
    for kind in ['Li2S', 'Li3N', 'LiTFSI']:
        ads0 = prepare_adsorbate(kind)
        if ads0 is None:
            continue

        out_db_path = OUT_DBS[kind]
        db_out = connect(out_db_path)

        rot_deg = ROT_Z_DEG[kind]

        for row in db_in.select():
            slab = row.toatoms()

            placed, actual_h = place_adsorbate_centered_auto(
                slab, ads0, kind,
                fx=TARGET_FRAC[0], fy=TARGET_FRAC[1],
                rot_z_deg=rot_deg
            )

            # 只在 x/y 上施加周期包裹（z 不包裹）
            placed.wrap(pbc=[True, True, False])

            # 写库：记录元数据（附上自动调整后的实际高度）
            db_out.write(
                placed,
                substrate=row.formula,
                adsorbate=kind,
                placement='center',
                height_A=actual_h,
                rot_z_deg=rot_deg,
                auto_close='vdW+fudge'
            )

        print(f"[OK] 已写出：{out_db_path}")

if __name__ == "__main__":
    main()


[OK] 已写出：Surfaces_Li2S.db
[OK] 已写出：Surfaces_Li3N.db
[OK] 已写出：Surfaces_LiTFSI.db
